# Step 5 · Estimate pathogen publications and calculate the PII

For each accepted pathogen–host pair, estimate disease-asserting publications as `(positive sampled abstracts / selected sample records) × pair-search records`. The denominator is the selected PMID sample (capped at 200 per pair). Every available abstract in the sample must be screened; the positive sample rate supplies the publication estimate. Divide that estimate by the organism-only PubMed total to calculate the pair-specific pathogenicity fraction. The displayed PII applies `log(1 + fraction)` and divides by the maximum transformed fraction within the pathogen group, producing a 0–1 group-normalized index. Count-only PubMed searches run only for pathogens accepted in at least one selected host.


## Step 1 · Load accepted pathogen–host pairs

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, ncbi_contact_email, staging_root_for_config
from pathogen_atlas_v4.runs import resolve_run
from pathogen_atlas_v4.pathogen_screening import build_pathogen_screen_tasks
from pathogen_atlas_v4.decision_runner import decision_cache_key_for_task
from pathogen_atlas_v4.pubmed_client import count_accepted_pathogen_publications, ncbi_api_key, literature_cache_key
from pathogen_atlas_v4.interest_index import build_pathogenic_interest_index
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('05_estimate_pathogen_publications', RUN_ID)
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_RECORD = json.loads((RUN_DIR / 'resolved_config.json').read_text(encoding='utf-8'))
CONFIG = RUN_RECORD['source_config']
RUN_MODE = str(CONFIG['run'].get('selection_mode', '')).casefold()
IS_FULL_RUN = RUN_MODE == 'full'
STAGING_TAXONOMY_ROOT = staging_root_for_config(ROOT, CONFIG) / 'taxonomy'
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
PATHOGENS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'hosts.parquet')
ALIASES = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogen_aliases.parquet')  # Keep name_type and mapping_status for safe alias filtering.
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)), known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)), host_group_ids={k:g.host_id.astype(str).tolist() for k,g in HOSTS.groupby('host_group')}, pathogen_group_ids={k:g.pathogen_id.astype(str).tolist() for k,g in PATHOGENS.groupby('pathogen_group')})
cache_key = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
                                  pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
                                  taxonomy_snapshot=STAGING_TAXONOMY_ROOT.parent.name,
                                  host_expansion_hash=CONFIG['acquisition'].get('host_expansion_hash', ''))
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
PAIR_SELECTION_PATH = RUN_DIR / 'pathogen_host_selection.parquet'
PAIR_SEARCHES_PATH = CACHE_DIR / 'tables/pair_searches.parquet'
SAMPLE_MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
SCREEN_LEDGER_PATH = ROOT / 'data/v4/cache/decisions/pathogen_screen_ledger.parquet'
COUNT_PATH = CACHE_DIR / 'tables/accepted_pathogen_publication_counts.parquet'
PII_PATH = RUN_DIR / 'release/pathogenic_interest_index.parquet'
if not PAIR_SELECTION_PATH.exists():
    raise FileNotFoundError(f'Run 04 and complete pathogen screening first: {PAIR_SELECTION_PATH}')
pair_selection = pd.read_parquet(PAIR_SELECTION_PATH)
if IS_FULL_RUN:
    if len(pair_selection) != len(resolved.pairs) or pair_selection.duplicated(['pathogen_id', 'host_id']).any():
        raise RuntimeError(f'Full-run pair selection does not cover all {len(resolved.pairs):,} configured pairs. Rerun 04 to completion.')
    ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
    if not PAIR_SEARCHES_PATH.exists() or not SAMPLE_MATCHES_PATH.exists() or not ARTICLES_PATH.exists():
        raise FileNotFoundError('Full-run pair searches, screening samples, and article cache must exist before PII release.')
    pair_searches = pd.read_parquet(PAIR_SEARCHES_PATH)
    selected_pair_keys = set((str(pathogen_id), str(host_id)) for pathogen_id, host_id in resolved.pairs)
    pair_search_index = pd.MultiIndex.from_frame(pair_searches[['pathogen_id', 'host_id']].astype(str))
    pair_searches = pair_searches.loc[pair_search_index.isin(selected_pair_keys)].copy()
    acquired_pair_keys = set(map(tuple, pair_searches.loc[pair_searches.status.astype(str).eq('completed'), ['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy()))
    missing_pair_searches = selected_pair_keys - acquired_pair_keys
    if missing_pair_searches:
        raise RuntimeError(f'Full-run pair acquisition is incomplete: {len(missing_pair_searches):,} configured pairs lack a completed PubMed search. Rerun 03_acquire_pair_pubmed_records.')
    sample_matches = pd.read_parquet(SAMPLE_MATCHES_PATH)
    if not sample_matches.empty:
        sample_pair_index = pd.MultiIndex.from_frame(sample_matches[['pathogen_id', 'host_id']].astype(str))
        sample_matches = sample_matches.loc[sample_pair_index.isin(selected_pair_keys)].copy()
    PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
    articles = pd.read_parquet(ARTICLES_PATH)
    screen_tasks = build_pathogen_screen_tasks(
        sample_matches, articles, PATHOGENS, HOSTS, PROMPTS, run_id=RUN_ID,
        run_config_version=str(CONFIG['run']['config_version']),
        model_version=str(CONFIG['decisions'].get('model_version', 'typesafe/jev-1.14')))
    required_task_keys = {decision_cache_key_for_task(task) for task in screen_tasks}
    search_pair_keys = set(map(tuple, pair_searches.loc[pd.to_numeric(pair_searches.search_count, errors='coerce').fillna(0).gt(0), ['pathogen_id', 'host_id']].astype(str).to_numpy()))
    expected_sample_pairs = selected_pair_keys & search_pair_keys
    prepared_sample_pairs = set(map(tuple, sample_matches[['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy())) if not sample_matches.empty else set()
    if expected_sample_pairs - prepared_sample_pairs:
        raise RuntimeError(f'Full-run sample cache misses {len(expected_sample_pairs - prepared_sample_pairs):,} PubMed-hit pairs. Rerun 04 sample preparation.')
    ledger = pd.read_parquet(SCREEN_LEDGER_PATH) if SCREEN_LEDGER_PATH.exists() else pd.DataFrame()
    if not ledger.empty and {'decision_kind', 'cache_key'}.issubset(ledger.columns):
        ledger = ledger.loc[ledger.decision_kind.astype(str).eq('pathogen_screen')
                            & ledger.cache_key.astype(str).isin(required_task_keys)]
    else:
        ledger = pd.DataFrame()
    completed_keys = set(ledger.loc[ledger.status.astype(str).eq('completed'), 'cache_key'].astype(str)) if not ledger.empty else set()
    missing_task_keys = required_task_keys - completed_keys
    if missing_task_keys:
        raise RuntimeError(f'Full-run screening is incomplete: {len(missing_task_keys):,} current abstract tasks remain unscreened. Finish 04 before PII release.')
    selected_pairs = set(map(tuple, pair_selection[['pathogen_id', 'host_id']].astype(str).to_numpy()))
    expected_pairs = set((str(pathogen_id), str(host_id)) for pathogen_id, host_id in resolved.pairs)
    if selected_pairs != expected_pairs:
        raise RuntimeError('Full-run pair selection does not match the configured pathogen–host pairs.')
    incomplete_pairs = pair_selection.loc[pair_selection.selection_status.astype(str).eq('incomplete')]
    if not incomplete_pairs.empty:
        screened = pd.to_numeric(incomplete_pairs.n_completed_screenings, errors='coerce').fillna(0)
        screenable = pd.to_numeric(incomplete_pairs.n_sampled_records_with_abstract, errors='coerce').fillna(0)
        unfinished_pairs = incomplete_pairs.loc[screened < screenable]
        if not unfinished_pairs.empty:
            raise RuntimeError(f'Full-run screening still has {len(unfinished_pairs):,} pairs with unscreened abstract-bearing records; rerun 04 before PII release.')
        print(f'Pairs marked incomplete only because sampled records lack abstracts: {len(incomplete_pairs):,}')
accepted_pairs = pair_selection.loc[pair_selection.selection_status.eq('accepted') & pair_selection.selected_as_pathogen_for_host.fillna(False).astype(bool)].copy()
accepted_pathogen_ids = sorted(accepted_pairs.pathogen_id.astype(str).unique())
print('Accepted pathogen–host pairs:', len(accepted_pairs), '| unique accepted pathogen IDs:', len(accepted_pathogen_ids))
print('Only these pathogen IDs are sent to the PubMed total-publication count query.')
LOG.info('PII preflight: accepted_pairs=%d accepted_pathogens=%d', len(accepted_pairs), len(accepted_pathogen_ids))

2026-11-03 16:38 INFO Notebook session started at 2026-11-03T16:38:55.024774+00:00
2026-11-03 16:38 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 16:38 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/05_estimate_pathogen_publications.log
2026-11-03 16:38 INFO PII preflight: accepted_pairs=348 accepted_pathogens=348


Pairs marked incomplete only because sampled records lack abstracts: 207
Accepted pathogen–host pairs: 348 | unique accepted pathogen IDs: 348
Only these pathogen IDs are sent to the PubMed total-publication count query.


## Step 2 · Count total organism publications for accepted pathogens only

These count-only searches provide the denominator. Each pathogen accepted for one or more hosts is queried once.


In [2]:
RUN_PUBMED_COUNT = IS_FULL_RUN  # Full mode completes missing accepted-pathogen counts; cached work is resumed.
if RUN_PUBMED_COUNT:
    NCBI_EMAIL = ncbi_contact_email(ROOT)
    NCBI_API_KEY = ncbi_api_key()
    counts = count_accepted_pathogen_publications(
        accepted_pathogen_ids, PATHOGENS, ALIASES, CACHE_DIR,
        query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
        email=NCBI_EMAIL, api_key=NCBI_API_KEY, logger=LOG)
    display(counts.head(20))
else:
    counts = pd.read_parquet(COUNT_PATH) if COUNT_PATH.exists() else pd.DataFrame()
    print('PubMed count queries skipped. Set RUN_PUBMED_COUNT=True to search only the accepted pathogen set.')
    print('Cached accepted-pathogen counts available:', len(counts))

Using the configured Git email as the NCBI contact address.


2026-11-03 16:39 INFO PubMed alias safeguard: {"aliases_missing_name": 0, "aliases_non_nomenclatural": 90177, "aliases_searchable": 36728, "aliases_shared_across_pathogens": 3953, "aliases_total": 332312, "aliases_unmapped": 21291, "aliases_vmr_records_checked": 36743, "aliases_vmr_terms_extracted": 65410}
2026-11-03 16:39 INFO Accepted pathogen publication counts 50/348
2026-11-03 16:39 INFO Accepted pathogen publication counts 100/348
2026-11-03 16:39 INFO Accepted pathogen publication counts 150/348
2026-11-03 16:40 INFO Accepted pathogen publication counts 200/348
2026-11-03 16:40 INFO Accepted pathogen publication counts 250/348
2026-11-03 16:40 INFO Accepted pathogen publication counts 300/348
2026-11-03 16:41 INFO Accepted pathogen publication counts 348/348


,pathogen_id,pathogen_group,query_hash,query_version,n_pubmed_search_records,status
0,bacteria:09e849d5c374f331c85d,parasites,f48c6322edce8d4ea6176e7940fe52ac46045a0d536212...,pathogen-vetted-aliases-title-abstract-v2,3,completed
1,bacteria:2112dd7bba0bead55eb1,parasites,ff6e13f9e50727777fa42ee646a87be016165767bb2b2f...,pathogen-vetted-aliases-title-abstract-v2,27,completed
2,bacteria:56faf32d2989cafe0289,parasites,f19030e056495f26634d793a439c0e1b18faf9bf1c4b19...,pathogen-vetted-aliases-title-abstract-v2,4,completed
3,bacteria:799700ac070f5c419589,parasites,ec4d115c9762f261a138f2adac34b88d52508e5df32a08...,pathogen-vetted-aliases-title-abstract-v2,4,completed
4,bacteria:94149d472b9198eef145,parasites,dcee88cdfa2887b12321b63a85cbe58bc24a56edd72de0...,pathogen-vetted-aliases-title-abstract-v2,1,completed
5,bacteria:bf17dab67ef0f6ca6ddc,parasites,8c46d99c04176c311fa5480f075f188ee3402f1ac82e56...,pathogen-vetted-aliases-title-abstract-v2,18,completed
6,bacteria:ec760985ce2241f77f1f,parasites,a709b5b4a2852fb2cf5708faeffe66f8cbcf5202d8261a...,pathogen-vetted-aliases-title-abstract-v2,2,completed
7,fungi:7718fd80a56ca847cbc0,parasites,0b1cc4de6717c911e350532f4a6557fd6240cf61c9c1d1...,pathogen-vetted-aliases-title-abstract-v2,93,completed
8,fungi:c5ac93664da523db6159,parasites,45e42ba0ba2c10598233f108de5628abdb93b43048f4e8...,pathogen-vetted-aliases-title-abstract-v2,16,completed
9,ncbi:1001829,parasites,eab6ff3ffe811f8a766018d80f3260c967186c3b6f2d7d...,pathogen-vetted-aliases-title-abstract-v2,18,completed


## Step 3 · Extrapolate pair evidence and calculate the PII


In [3]:
if accepted_pathogen_ids and counts.empty:
    raise FileNotFoundError(f'Run accepted-pathogen PubMed count queries first: {COUNT_PATH}')
pii = build_pathogenic_interest_index(counts, pair_selection, PATHOGENS)
RUN_DIR.joinpath('release').mkdir(parents=True, exist_ok=True)
pii.to_parquet(PII_PATH, index=False)
print('Saved:', PII_PATH)
if pii.empty:
    print('No accepted pathogen–host pairs; no PII rows are emitted.')
else:
    display(pii.sort_values('pathogenic_interest_index', ascending=False).head(25))
LOG.info('PII table saved: accepted_pairs=%d accepted_pathogens=%d rows=%d path=%s',
         len(accepted_pairs), len(accepted_pathogen_ids), len(pii), PII_PATH)


Saved: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/release/pathogenic_interest_index.parquet


,pathogen_id,host_id,accepted_name,pathogen_group,n_positive_records,n_sampled_records,n_abstracts_screened,n_pair_search_records,estimated_pathogen_publications,estimated_pathogen_publications_index,n_pubmed_search_records,pathogenicity_fraction,pathogenicity_fraction_index,pathogenic_interest_index,index_version,query_hash,query_version
64,ncbi:1582908,ncbi:9606,Leishmania waltoni,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,f46e2f7cd785f7d32bcdb8606cee8f0cc85c1201860be5...,pathogen-vetted-aliases-title-abstract-v2
161,ncbi:3081638,ncbi:9606,Leishmania ellisi,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,fbc284e3cf5c6004cbd550994b987ca7c68a182a41bd4c...,pathogen-vetted-aliases-title-abstract-v2
47,ncbi:1348868,ncbi:9606,Tetratrichomonas empyemagena,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,e4ab6a4a51785e8bd1fb9b83a86d3971348e85f194963d...,pathogen-vetted-aliases-title-abstract-v2
312,ncbi:65664,ncbi:9606,Acanthamoeba stevensoni,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,f1ccf2c2d1be3393070aead3a3fc84ad666f73bd85426b...,pathogen-vetted-aliases-title-abstract-v2
163,ncbi:3082057,ncbi:9606,Philophthalmus hechingeri,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,b2543bfc221cd0db36894cadfb093409ec985dce75e33b...,pathogen-vetted-aliases-title-abstract-v2
111,ncbi:2506114,ncbi:9606,Acanthamoeba byersi,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,fe21df629c45a35952d79b5b4dd82a5da6be903b45306f...,pathogen-vetted-aliases-title-abstract-v2
328,ncbi:767498,ncbi:9606,Acanthamoeba genotype T13,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,2af8be33bdd4c58c4b525411dce8e07a86a09a0f20e744...,pathogen-vetted-aliases-title-abstract-v2
144,ncbi:2936577,ncbi:9606,Cryptosporidium sp. 1,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,df671572122079693ed47c54204990846c9ef538051cc9...,pathogen-vetted-aliases-title-abstract-v2
4,bacteria:94149d472b9198eef145,ncbi:9606,Yersinia canariae,parasites,1,1,1,1,1.000,0.084300,1,1.000000,1.000000,1.000000,pathogenic-interest-index-v4.5-volume-and-frac...,dcee88cdfa2887b12321b63a85cbe58bc24a56edd72de0...,pathogen-vetted-aliases-title-abstract-v2
78,ncbi:1856844,ncbi:9606,Haycocknema perplexum,parasites,9,11,11,11,9.000,0.280038,11,0.818182,0.862496,0.862496,pathogenic-interest-index-v4.5-volume-and-frac...,eda2bcbf70ab3066c36b566e990a90eb2226bd29d42f58...,pathogen-vetted-aliases-title-abstract-v2


2026-11-03 16:41 INFO PII table saved: accepted_pairs=348 accepted_pathogens=348 rows=348 path=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/release/pathogenic_interest_index.parquet


## Interpretation

The sample numerator is abstracts that explicitly support disease caused by this pathogen in the selected host. Divide that count by the full selected PMID sample, then extrapolate across all pair-search records. Every available abstract must be screened before the estimate is produced. Divide the estimated count by the organism-only PubMed total to get the pair-specific pathogenicity fraction. The final PII is `log(1 + fraction)` divided by the maximum transformed fraction within the pathogen group, so it ranges from 0 to 1 and is normalized separately by group. The output retains the selected sample size, screened-abstract count, pair-search count, estimated pathogenicity-positive publications, and organism-publication denominator for audit.
